# Reproduce on Colab (L4 GPU)

Runtime → Change runtime type → L4 GPU. The full rerun takes several hours, so Colab Pro (background execution) is needed for `full.sh`; the smoke test takes under 10 minutes.

ImageNet needs a Hugging Face token with access to `ILSVRC/imagenet-1k` (add it as the Colab secret `HF_TOKEN`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the code
REPO = 'https://github.com/hhuang999/vit-fastweight-lens.git'
!git clone -q --branch release/v1 $REPO /content/vit-fastweight-lens
%cd /content/vit-fastweight-lens
!git log --oneline -1

In [ ]:
# Pinned environment (Python 3.12 on Colab): CUDA 12.8 PyTorch, then the locked requirements and the package
!pip install -q -r requirements-torch.txt --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!pip install -q gdown

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # never printed

In [ ]:
# Data (about 15 minutes) and models; prepare_data.py --check verifies counts and fixed lists
!python scripts/prepare_data.py imagenet imagenet_s clickme
!python scripts/prepare_data.py --check
!python scripts/download_models.py

In [ ]:
!python -m pytest -q

In [ ]:
# Smoke test: every experiment on 16 images (under 10 minutes)
!bash reproduce/smoke.sh

In [ ]:
# Full rerun (several GPU-hours on an L4); completed runs are skipped on restart
!bash reproduce/full.sh 2>&1 | tee full.log

In [ ]:
# Tables and figures from the rerun outputs (also done at the end of full.sh; without an argument, from the committed outputs/)
!bash reproduce/from_outputs.sh outputs/rerun
!cat paper/tables_own/main_results.tex